### Import Libraries and data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
def reduce_mem_usage(df):

    start_mem = df.memory_usage().sum() 
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() 
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [3]:
sample_feature = reduce_mem_usage(pd.read_csv('data_for_tree.csv'))

Memory usage of dataframe is 63691972.00 MB
Memory usage after optimization is: 17117446.00 MB
Decreased by 73.1%


In [4]:
continuous_feature_names = [x for x in sample_feature.columns if x not in ['price','brand','model','brand']]

In [5]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [6]:
# base modules
from pathlib import Path
import logging
from collections import OrderedDict

# for manipulating data
import numpy as np
import pandas as pd
import math

# for Machine Learning
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

# for visualization
from matplotlib import pyplot as plt

In [7]:
from typing import Callable
import copy
import re
import numpy as np
import pandas as pd
from pandas.api.types import is_string_dtype, is_numeric_dtype
def process_df(
    df: pd.DataFrame, 
    y_field: str | None = None, 
    skip_flds: list = [],
    ignore_flds: list = [], 
    na_dict: dict = {},
    preproc_fn: Callable = None, 
    max_n_cat = None, 
    ):
    """
    Take a dataframe df and splits off the response variable, and
    changes the df into an entirely numeric dataframe. For each column of df 
    which is not in skip_flds nor in ignore_flds, na values are replaced by the
    median value of the column.
    """
    df = copy.deepcopy(df)
    
    df_ignored = df.loc[:, ignore_flds]
    df = df.drop(columns = ignore_flds)
    
    if preproc_fn: 
        preproc_fn(df)
        
    if y_field is None: 
        y = None
    else:
        if not is_numeric_dtype(df[y_field]): 
            df[y_field] = pd.Categorical(df[y_field]).codes
        y = df[y_field].values
        skip_flds += [y_field]
    
    df = df.drop(columns = skip_flds)

    na_dict_initial = na_dict.copy()
    for n, c in df.items(): 
        na_dict = fix_missing(df, c, n, na_dict)
    
    if len(na_dict_initial) > 0:
        df = df.drop(
            [a + '_na' for a in list(set(na_dict.keys()) - set(na_dict_initial.keys()))], 
            axis = 1,
        )
    for n,c in df.items(): 
        df = numericalize(df, c, n, max_n_cat)
        
    df = pd.get_dummies(df, dummy_na = True)
    df = pd.concat([df_ignored, df], axis = 1)
    return (df, y, na_dict)


In [8]:
def fix_missing(df, col, name, na_dict):
    """
    Fill missing data in a column of df with the median, and add a {name}_na column
    which specifies if the data was missing.
    """
    if is_numeric_dtype(col):
        if pd.isnull(col).sum() or (name in na_dict):
            df[name + '_na'] = pd.isnull(col)
            filler = na_dict[name] if name in na_dict else col.median()
            df[name] = col.fillna(filler)
            na_dict[name] = filler
    return na_dict

    
def numericalize(df: pd.DataFrame, col: str, name: str, max_n_cat: int | None) -> pd.DataFrame:
    """
    Changes the column col from a categorical type to it's integer codes.
    """
    df = copy.deepcopy(df)
    if (not is_numeric_dtype(col) 
        and (max_n_cat is None or len(col.cat.categories) > max_n_cat)):
        df[name] = pd.Categorical(col).codes + 1
    return df

In [9]:
df, y, nas = process_df(sample_feature, 'price')

In [10]:
print(df.shape)
df.head()

(199037, 48)


,SaleID,name,model,brand,bodyType,fuelType,gearbox,power,kilometer,notRepairedDamage,...,power_bin,model_na,bodyType_na,fuelType_na,gearbox_na,train_na,test_na,used_time_na,city_na,power_bin_na
0,0,736,30.0,6,1.0,0.0,0.0,60,12.5,2,...,5.0,False,False,False,False,False,True,False,False,False
1,1,2262,40.0,1,2.0,0.0,0.0,0,15.0,1,...,11.0,False,False,False,False,False,True,False,False,True
2,2,14874,115.0,15,1.0,0.0,0.0,163,12.5,2,...,16.0,False,False,False,False,False,True,False,False,False
3,3,71865,109.0,10,0.0,0.0,1.0,193,15.0,2,...,19.0,False,False,False,False,False,True,False,True,False
4,4,111080,110.0,5,1.0,0.0,0.0,68,5.0,2,...,6.0,False,False,False,False,False,True,False,False,False


In [11]:
len(y)
y

array([1850., 3600., 6222., ...,   nan,   nan,   nan],
      shape=(199037,), dtype=float32)

In [12]:
import numpy as np
import pandas as pd

# 假设 y 可能是 Series 或 ndarray
y = pd.to_numeric(y, errors='coerce')  # 把非数值强制转 NaN

# 如果是 ndarray，则用 np.isnan 来处理掩码
if isinstance(y, np.ndarray):
    mask = ~np.isnan(y)
else:
    mask = y.notna()

# 过滤掉 y 的缺失样本
X = df.loc[mask].copy()
y = y[mask].astype(float).ravel()

# 处理 X 中可能残留的 NaN
if X.isna().any().any():
    X = X.fillna(X.median(numeric_only=True))

In [13]:
y_series = pd.to_numeric(pd.Series(y), errors='coerce')
mask = y_series.notna()

X = X.loc[mask].copy()
y = y_series.loc[mask].astype(float).to_numpy().ravel()

# 2) 若 X 仍有 NaN（比如有 skip_flds），再做一次填补
if X.isna().any().any():
    X = X.fillna(X.median(numeric_only=True))

# 3) 训练前自检
print("X rows:", X.shape[0], "y rows:", y.shape[0], "X cols:", X.shape[1])
assert X.shape[0] == y.shape[0]

# 4) 训练（注意用 X，不是原 df）
model = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
model.fit(X, y)

X rows: 149037 y rows: 149037 X cols: 48


,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
x = df.iloc[0]
y_true = np.exp(y[0])
x, y_true

(SaleID                          0
 name                          736
 model                        30.0
 brand                           6
 bodyType                      1.0
 fuelType                      0.0
 gearbox                       0.0
 power                          60
 kilometer                    12.5
 notRepairedDamage               2
 seller                          0
 offerType                       0
 v_0                      43.34375
 v_1                      3.966797
 v_2                      0.050262
 v_3                      2.160156
 v_4                      1.143555
 v_5                      0.235718
 v_6                       0.10199
 v_7                      0.129517
 v_8                      0.022812
 v_9                      0.097473
 v_10                    -2.880859
 v_11                     2.804688
 v_12                    -2.419922
 v_13                      0.79541
 v_14                     0.914551
 train                         1.0
 test               

In [15]:
x = [x]                   # wrap the data point into a list
y_pred = model.predict(x) # prediction, as list
y_pred = y_pred[0]        # take first element of predicted list
y_pred = np.exp(y_pred)   # exponentialize
y_pred

np.float64(inf)

In [16]:
model.score(X, y)

0.9950527226149228

In [17]:
n_total = len(df)
n_valid = 50000      
n_small = 60000      # 小样本集大小（快速实验）
n_train = n_total - n_valid

In [18]:
def split_vals(df: pd.DataFrame, n: int) -> pd.DataFrame: 
    return df[:n].copy(), df[n:].copy()

In [19]:
X_train, X_valid = split_vals(df, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small, _ = split_vals(X_train, n_small)
y_small, _ = split_vals(y_train, n_small)

print('Number of small training data points: X = {}, y = {}'.format(X_small.shape, y_small.shape))
print('Number of full training data points: X = {}, y = {}'.format(X_train.shape, y_train.shape))
print('Number of validation data points: X = {}, y = {}'.format(X_valid.shape, y_valid.shape))

Number of small training data points: X = (60000, 48), y = (60000,)
Number of full training data points: X = (149037, 48), y = (149037,)
Number of validation data points: X = (50000, 48), y = (0,)


In [20]:
n_total   = len(df)          
n_test= n_total - len(y)  
X_labeled = df.iloc[: n_total - n_test].copy() 


# ---------- 2) 在“有标签段”里做最简单切分 ----------
n_valid = 12000          # 老师例子
n_train = len(X_labeled) - n_valid

def split_vals(a, n):
    return a[:n].copy(), a[n:].copy()

X_train, X_valid = split_vals(X_labeled, n_train)
y_train, y_valid = split_vals(y,          n_train)

In [21]:
def rmse(y_gold, y_pred): 
    return math.sqrt(((y_gold - y_pred)**2).mean())

In [22]:
def print_score(m, X_train, y_train, X_valid, y_valid):
    print('RMSE on train set: {:.4f}'.format(rmse(m.predict(X_train), y_train)))
    print('RMSE on valid set: {:.4f}'.format(rmse(m.predict(X_valid), y_valid)))
    print('R^2 on train set: {:.4f}'.format(m.score(X_train, y_train)))
    print('R^2 on valid set: {:.4f}'.format(m.score(X_valid, y_valid)))
    if hasattr(m, 'oob_score_'): print('R^2 on oob set: {:.4f}'.format(m.oob_score_))
    return

In [23]:
base_model = RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=42)
base_model.fit(X_small, y_small)
print_score(base_model, X_small, y_small, X_valid, y_valid)

RMSE on train set: 628.2214
RMSE on valid set: 1488.8488
R^2 on train set: 0.9918
R^2 on valid set: 0.9542


In [24]:
base_model = RandomForestRegressor(n_estimators = 10, n_jobs = -1, random_state = 42)

%time base_model.fit(X_train, y_train)
print_score(base_model, X_train, y_train, X_valid, y_valid)

CPU times: total: 37 s
Wall time: 4.54 s
RMSE on train set: 610.5800
RMSE on valid set: 1481.9888
R^2 on train set: 0.9924
R^2 on valid set: 0.9547


In [25]:
model_dt = DecisionTreeRegressor(
    criterion = 'squared_error',
    splitter = 'best', 
    max_depth = 3,
    min_samples_split = 2, 
    min_samples_leaf = 1,
    min_weight_fraction_leaf = 0.0, 
    max_features = None, 
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0, 
    ccp_alpha = 0.0,
    random_state = 42, 
)

In [26]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

CPU times: total: 281 ms
Wall time: 289 ms
RMSE on train set: 2908.8824
RMSE on valid set: 3036.8245
R^2 on train set: 0.8251
R^2 on valid set: 0.8096


In [27]:
model_dt = DecisionTreeRegressor(
    criterion = 'squared_error',
    splitter = 'best', 
    max_depth = 15,
    min_samples_split = 2, 
    min_samples_leaf = 1,
    min_weight_fraction_leaf = 0.0, 
    max_features = None, 
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0, 
    ccp_alpha = 0.0,
    random_state = 42,
)

In [28]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

CPU times: total: 1.36 s
Wall time: 1.41 s
RMSE on train set: 625.8119
RMSE on valid set: 2039.4831
R^2 on train set: 0.9919
R^2 on valid set: 0.9141


In [29]:
model_dt = DecisionTreeRegressor(
    criterion = 'squared_error',
    splitter = 'best', 
    max_depth = 15,
    min_samples_split = 2, 
    min_samples_leaf = 1,
    min_weight_fraction_leaf = 0.0, 
    max_features = None, 
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0001, 
    ccp_alpha = 0.0,
    random_state = 42,
)

In [30]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

CPU times: total: 1.34 s
Wall time: 1.37 s
RMSE on train set: 625.8119
RMSE on valid set: 2039.4830
R^2 on train set: 0.9919
R^2 on valid set: 0.9141


In [31]:
model_dt = DecisionTreeRegressor(
    criterion = 'squared_error',
    splitter = 'best', 
    max_depth = 3,
    min_samples_split = 2, 
    min_samples_leaf = 1,
    min_weight_fraction_leaf = 0.0, 
    max_features = None, 
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0, 
    ccp_alpha = 0.0,
    random_state = 42,
)

In [32]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

CPU times: total: 250 ms
Wall time: 282 ms
RMSE on train set: 2908.8824
RMSE on valid set: 3036.8245
R^2 on train set: 0.8251
R^2 on valid set: 0.8096


In [33]:
import re
import IPython
import graphviz
from sklearn.base import ClassifierMixin
from sklearn.tree import export_graphviz
import matplotlib.pyplot as plt


def set_plot_sizes(sml: int, med: int, big: int) -> None:
    plt.rc('font', size = sml)          # controls default text sizes
    plt.rc('axes', titlesize = sml)     # fontsize of the axes title
    plt.rc('axes', labelsize = med)     # fontsize of the x and y labels
    plt.rc('xtick', labelsize = sml)    # fontsize of the tick labels
    plt.rc('ytick', labelsize = sml)    # fontsize of the tick labels
    plt.rc('legend', fontsize = sml)    # legend fontsize
    plt.rc('figure', titlesize = big)   # fontsize of the figure title
    return

    
def draw_tree(
    tree: ClassifierMixin, 
    feature_names: list[str], 
    size: int = 10, 
    ratio: float = 0.6, 
    precision: int = 0,
    ):
    """
    Draws a representation of a random forest in IPython.
    """
    s = export_graphviz(
        tree, 
        out_file = None, 
        feature_names = feature_names, 
        filled = True, 
        special_characters = True, 
        rotate = True, 
        precision = precision,
    )
    # return s
    IPython.display.display(
        graphviz.Source(re.sub('Tree {', f'Tree {{ size={size}; ratio={ratio}', s))
    )

In [34]:
fig, axes = plt.subplots(nrows = 1, ncols = 1, figsize = (15, 10), dpi = 300)
plot_tree(model_dt, filled = True)

[Text(0.5, 0.875, 'x[12] <= 46.516\nsquared_error = 48367526.881\nsamples = 60000\nvalue = 5752.978'),
 Text(0.25, 0.625, 'x[15] <= -0.725\nsquared_error = 10729296.471\nsamples = 49756\nvalue = 3493.891'),
 Text(0.375, 0.75, 'True  '),
 Text(0.125, 0.375, 'x[24] <= 1.981\nsquared_error = 11203124.56\nsamples = 12674\nvalue = 7483.353'),
 Text(0.0625, 0.125, 'squared_error = 5336603.168\nsamples = 8566\nvalue = 6083.292'),
 Text(0.1875, 0.125, 'squared_error = 10825698.828\nsamples = 4108\nvalue = 10402.761'),
 Text(0.375, 0.375, 'x[12] <= 44.422\nsquared_error = 3268380.11\nsamples = 37082\nvalue = 2130.36'),
 Text(0.3125, 0.125, 'squared_error = 1104152.219\nsamples = 25664\nvalue = 1417.748'),
 Text(0.4375, 0.125, 'squared_error = 4425941.462\nsamples = 11418\nvalue = 3732.084'),
 Text(0.75, 0.625, 'x[24] <= 3.997\nsquared_error = 85994151.906\nsamples = 10244\nvalue = 16725.558'),
 Text(0.625, 0.75, '  False'),
 Text(0.625, 0.375, 'x[24] <= 2.487\nsquared_error = 33648293.478\nsamp

In [35]:
draw_tree(model_dt, feature_names = X_small.columns.tolist(), precision = 3)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [36]:
model_rf = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators = 1, # = the model is a single tree
    criterion = 'squared_error', 
    max_depth = 3,
    min_samples_split = 2, 
    min_samples_leaf = 1,
    min_weight_fraction_leaf = 0.0, 
    max_features = None, 
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0,
    ccp_alpha = 0.0, 
    random_state = 42,
    
    # RandomForestRegressor specific hyperparameters
    bootstrap = False, # default = True 
    oob_score = False, 
    max_samples = None,

    # extra parameters
    warm_start = False, 
    n_jobs = -1,
    verbose = 0, 
)

In [37]:
%time model_rf.fit(X_small, y_small)
print_score(model_rf, X_small, y_small, X_valid, y_valid)

CPU times: total: 266 ms
Wall time: 299 ms
RMSE on train set: 2908.8824
RMSE on valid set: 3036.8245
R^2 on train set: 0.8251
R^2 on valid set: 0.8096


In [38]:
draw_tree(model_rf.estimators_[0], X_small.columns.tolist(), precision = 3)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH